# 01 Azure Cloud dla QA - podstawy i srodowisko pracy

_Kamil Bartocha_ | wersja 1.0

## Rozklad jazdy

1. 🔹 Chmura obliczeniowa - IaaS / PaaS / SaaS
2. 🔹 Hierarchia Azure - tenant, subskrypcja, resource group
3. 🔹 Azure CLI - podstawy i automatyzacja
4. 🔹 Regiony i strefy dostepnosci
5. 🔹 Tagowanie zasobow dla srodowisk testowych

---

## 1. 🔹 Chmura obliczeniowa - IaaS / PaaS / SaaS

Chmura obliczeniowa (cloud computing) to model dostarczania zasobow
informatycznych przez internet - serwerow, baz danych, sieci,
oprogramowania - na zadanie, z platnoscia za faktycznie uzyte zasoby.

Rozrozniamy trzy glowne modele serwisow:

**IaaS (Infrastructure as a Service)** - wynajmujemy wirtualne maszyny,
sieci i przestrzen dyskowa. Sami zarzadzamy systemem operacyjnym,
middleware i aplikacja. Przyklad: Azure Virtual Machines, Azure VNet.

**PaaS (Platform as a Service)** - dostawca zarzadza infrastruktura
i platforma. My dostarczamy tylko kod i dane aplikacji.
Przyklad: Azure App Service, Azure SQL Database, Azure Functions.

**SaaS (Software as a Service)** - uzywamy gotowej aplikacji przez
przegladarke lub klienta. Dostawca zarzadza calym stosem.
Przyklad: Microsoft 365, GitHub (w czesci SaaS).

| Model | My zarzadzamy        | Dostawca zarzadza          | Przyklad Azure         |
|-------|----------------------|----------------------------|------------------------|
| IaaS  | OS, middleware, app  | serwery, siec, dyski       | Virtual Machines, VNet |
| PaaS  | aplikacja, dane      | OS, runtime, infrastruktura| App Service, Functions |
| SaaS  | dane uzytkownikow    | wszystko                   | Microsoft 365          |

> 💡 Wiekszoc systemow, ktore testujemy jako QA, dziala w modelu
> PaaS (Azure Functions, App Service, Service Bus, SQL). Znajomosc
> modelu pomaga okreslic, gdzie konczy sie odpowiedzialnosc zespolu
> deweloperskiego, a gdzie zaczyna sie infrastruktura dostawcy.

In [ ]:
import subprocess
import json


def az(command, output="json"):
    """Run Azure CLI command and return parsed JSON or raw text."""
    full_cmd = ["az"] + command.split() + ["--output", output]
    result = subprocess.run(full_cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Azure CLI error: {result.stderr.strip()}")
    if output == "json":
        return json.loads(result.stdout)
    return result.stdout.strip()


# Sprawdzamy aktywne konto - az account show
account = az("account show")
print(f"Subskrypcja:  {account['name']}")
print(f"Tenant ID:    {account['tenantId']}")
print(f"Stan:         {account['state']}")
print(f"Srodowisko:   {account['environmentName']}")

---

### 🐍 Cwiczenia - chmura obliczeniowa

**1.** Uruchom `az account show` i wypisz nazwe subskrypcji oraz
jej ID (`id`) w formacie: `Subskrypcja: <nazwa> (ID: <id>)`.

**2.** Dane `SERVICES` zawieraja liste uslug Azure. Uzupelnij slownik
`classified` przypisujac kazda usluge do modelu IaaS, PaaS lub SaaS.

**3.** Sprawdz czy aktywna subskrypcja jest w stanie `Enabled`.
Wypisz komunikat: `OK - subskrypcja aktywna` lub
`UWAGA - stan: <stan>`. *(Trudniejsze)*

In [ ]:
# Cwiczenie 1 - az account show - nazwa i ID subskrypcji
account = az("account show")
name = ...
sub_id = ...

print(f"Subskrypcja: {name} (ID: {sub_id})")

In [ ]:
# Cwiczenie 2 - klasyfikacja uslug Azure
SERVICES = [
    "Azure Virtual Machines",
    "Azure App Service",
    "Microsoft 365",
    "Azure Functions",
    "Azure Virtual Network",
    "Azure SQL Database",
    "Azure Blob Storage",
    "Azure Kubernetes Service",
]

# hint: sklasyfikuj recznie - przypisz kazda usluge do jednego z kluczy
classified = {
    "IaaS": [...],
    "PaaS": [...],
    "SaaS": [...],
}

for model, services in classified.items():
    print(f"{model}: {', '.join(services)}")

In [ ]:
# Cwiczenie 3 - sprawdzenie stanu subskrypcji *(Trudniejsze)*
# hint: uzyj account['state'] i porownaj z 'Enabled'
account = az("account show")
state = ...

if ...:
    print("OK - subskrypcja aktywna")
else:
    print(f"UWAGA - stan: {state}")

---

## 2. 🔹 Hierarchia Azure - tenant, subskrypcja, resource group

Zasoby w Azure sa zorganizowane w hierarchiczna strukture:

```
Tenant (organizacja w AAD)
 └── Subskrypcja (subscription) - jednostka rozliczeniowa
      └── Grupa zasobow (resource group) - logiczny kontener
           └── Zasob (resource) - VM, baza, kolejka, funkcja...
```

**Tenant** to instancja Azure Active Directory reprezentujaca
organizacje. Jeden tenant moze miec wiele subskrypcji.

**Subskrypcja (subscription)** to jednostka rozliczeniowa i granicy
uprawnien. Koszty sa agregowane na poziomie subskrypcji.

**Grupa zasobow (resource group)** to logiczny kontener na powiazane
zasoby. Usuwajac resource group, usuwamy wszystkie zasoby w niej.
To kluczowy mechanizm dla QA - tworzenie izolowanych srodowisk
testowych i ich czyszczenie jednym poleceniem.

**Zasob (resource)** to konkretna usluga: maszyna wirtualna, baza
danych, konto storage, kolejka Service Bus, itp.

> 💡 Dla QA: kazde srodowisko testowe powinno miec wlasna resource
> group (np. `rg-qa-pr-123`). Po tescie robimy `az group delete` i
> srodowisko znika bez pozostalosci. To fundament izolacji testow.

In [ ]:
# Hierarchia Azure - tenant i subskrypcja
account = az("account show")
print("=== Aktywne konto ===")
print(f"Tenant ID:      {account['tenantId']}")
print(f"Subskrypcja:    {account['name']}")
print(f"Subscription ID:{account['id']}")
print()

# Lista wszystkich resource groups w subskrypcji
groups = az("group list")
print(f"=== Resource groups ({len(groups)}) ===")
for rg in groups:
    print(f"  {rg['name']:<40} {rg['location']}")

---

### 🐍 Cwiczenia - hierarchia Azure

**1.** Wypisz Tenant ID i Subscription ID z `az account show`.

**2.** Wypisz wszystkie resource groups w formie tabeli:
`Nazwa | Lokalizacja | Stan (provisioningState)`.

**3.** Policz ile resource groups jest w kazdej lokalizacji
i wypisz zestawienie. *(Trudniejsze)*

In [ ]:
# Cwiczenie 1 - tenant ID i subscription ID
account = az("account show")
tenant_id = ...
subscription_id = ...

print(f"Tenant ID:       {tenant_id}")
print(f"Subscription ID: {subscription_id}")

In [ ]:
# Cwiczenie 2 - tabela resource groups
groups = az("group list")
print(f"{'Nazwa':<40} {'Lokalizacja':<20} Stan")
print("-" * 72)
for rg in groups:
    name = ...
    location = ...
    state = ...
    print(f"{name:<40} {location:<20} {state}")

In [ ]:
# Cwiczenie 3 - resource groups per lokalizacja *(Trudniejsze)*
# hint: uzyj slownika {location: count}
groups = az("group list")
location_counts = {}

for rg in groups:
    location = ...
    ...

for location, count in sorted(location_counts.items()):
    print(f"{location}: {count}")

---

## 3. 🔹 Azure CLI - podstawy i automatyzacja

Azure CLI (`az`) to oficjalny klient wiersza polecen do zarzadzania
zasobami Azure. Instalujemy go lokalnie lub uzywamy w Azure Cloud Shell.

**Struktura komendy:**
```
az <zasob> <akcja> [parametry]
az group  create  --name rg-qa-lab --location polandcentral
az group  list
az group  delete  --name rg-qa-lab --yes
```

**Najwazniejsze komendy dla QA:**

| Komenda                    | Opis                              |
|----------------------------|-----------------------------------|
| `az login`                 | logowanie interaktywne            |
| `az account show`          | aktywna subskrypcja               |
| `az account list`          | wszystkie subskrypcje             |
| `az group list`            | wszystkie resource groups         |
| `az group create`          | tworzenie resource group          |
| `az group delete`          | usuniecie resource group          |
| `az resource list`         | zasoby (opcjonalnie w grupie)     |
| `az version`               | wersja CLI                        |

**Formaty wyjscia:** `--output json` (domyslny), `--output table`,
`--output tsv` (dobre do parsowania w skryptach).

**Filtrowanie z JMESPath:** parametr `--query` pozwala wybrac
konkretne pola bez koniecznosci parsowania w Pythonie:
```bash
az group list --query "[].name" --output tsv
az group list --query "[?location=='westeurope'].name" --output tsv
```

> 💡 Dla automatyzacji testow: uzywamy `az` przez `subprocess` w
> Pythonie. Funkcja pomocnicza `az()` z tego modulu pozwala pisac
> czytelne skrypty bez recztnego parsowania wyjscia CLI.

In [ ]:
# Wersja Azure CLI - az version
version_info = az("version")
print(f"Azure CLI:     {version_info['azure-cli']}")
print(f"Rozszerzenia:  {list(version_info.get('extensions', {}).keys())}")

In [ ]:
# Format wyjscia - tabela (czytelny dla czlowieka)
print(az("group list", output="table"))

In [ ]:
# Filtrowanie z --query (JMESPath) - tylko nazwy resource groups
cmd = "group list --query [].name"
names = az(cmd)
print("Nazwy resource groups:")
for name in names:
    print(f"  - {name}")

In [ ]:
# Tworzenie i usuwanie resource group
rg_name = "rg-qa-demo"
location = "westeurope"

# Tworzenie resource group
result = az(f"group create --name {rg_name} --location {location}")
print(f"Utworzono: {result['name']} w {result['location']}")
print(f"Stan:      {result['properties']['provisioningState']}")

In [ ]:
# Usuwanie resource group (--yes pomija potwierdzenie)
az(f"group delete --name {rg_name} --yes")
print(f"Resource group '{rg_name}' usunieta.")

---

### 🐍 Cwiczenia - Azure CLI

**1.** Wypisz wersje Azure CLI z `az version` w formacie
`Azure CLI: x.y.z`.

**2.** Pobierz liste resource groups uzywajac `--query` tak, by
wynik zawierial tylko pola `name` i `location` (lista slownikow).

**3.** Uzyj `--output tsv` i `--query` do pobrania samych nazw
resource groups jako tekstu. Podziel tekst na linie i wypisz je.

**4.** Napisz funkcje `resource_group_exists(name)`, ktora zwraca
`True` jesli resource group o podanej nazwie istnieje. *(Trudniejsze)*

In [ ]:
# Cwiczenie 1 - wersja Azure CLI
version_info = az("version")
cli_version = ...

print(f"Azure CLI: {cli_version}")

In [ ]:
# Cwiczenie 2 - --query dla pol name i location
# hint: w JMESPath wybieramy pola przez [].{name:name,location:location}
groups = az("group list --query [].{name:name,location:location}")

for rg in groups:
    print(f"{rg['name']:<40} {rg['location']}")

In [ ]:
# Cwiczenie 3 - output tsv, podzial na linie
raw = az("group list --query [].name", output="tsv")
names = ...

print(f"Znalezione resource groups ({len(names)}):")
for name in names:
    print(f"  {name}")

In [ ]:
# Cwiczenie 4 - resource_group_exists(name) *(Trudniejsze)*
# hint: pobierz liste nazw i sprawdz czy name jest w liscie
def resource_group_exists(name):
    ...


print(resource_group_exists("rg-qa-lab"))    # True lub False
print(resource_group_exists("rg-nie-istnieje"))  # False

---

## 4. 🔹 Regiony i strefy dostepnosci

**Region (region)** to geograficznie wyodrebniony zbior datacenter.
Azure ma ponad 60 regionow na swiecie. Wybor regionu wplywa na:
- **latencje** - testowane systemy powinny byc w tym samym regionie
  co narzedzia testowe
- **zgodnosc z przepisami (compliance)** - dane medyczne, finansowe
  moga byc przechowywane tylko w okreslonych regionach
- **dostepnosc uslug** - nie kazdy region wspiera wszystkie uslugi
- **koszty** - ceny roznia sie miedzy regionami

**Para regionow (region pair)** to dwa regiony w tej samej geografii
polaczone na potrzeby Disaster Recovery. Jesli jeden region upadnie,
replikowane dane sa dostepne w parze. Przyklad: Poland Central
jest parowany z West Europe.

**Strefa dostepnosci (availability zone, AZ)** to fizycznie
oddzielone datacenter w tym samym regionie (oddzielne zasilanie,
chlodzenie, siec). Umozliwia budowanie systemow odpornych na awarie
jednego datacenter.

| Pojecie            | Zasieg    | Cel                                 |
|--------------------|-----------|-------------------------------------|
| Region             | ~kraj     | podstawowa jednostka wdrozenia      |
| Para regionow      | ~kontynent| replikacja DR miedzy regionami      |
| Availability Zone  | miasto    | HA w jednym regionie                |

> 💡 Dla QA: srodowiska testowe tworzymy w tym samym regionie co
> produkcja - unikamy sztucznej latencji miedzyregionalnej.
> Preferujemy regiony europejskie (Poland Central, West Europe,
> North Europe) ze wzgledu na RODO i czas odpowiedzi.

In [ ]:
# Lista dostepnych regionow - az account list-locations
locations = az("account list-locations")
print(f"Laczna liczba regionow: {len(locations)}")
print()

# Filtrujemy regiony europejskie po displayName
european = [
    loc for loc in locations
    if "europe" in loc["displayName"].lower()
    or "poland" in loc["displayName"].lower()
]

print(f"Regiony europejskie ({len(european)}):")
for loc in sorted(european, key=lambda x: x["displayName"]):
    print(f"  {loc['name']:<25} {loc['displayName']}")

---

### 🐍 Cwiczenia - regiony

**1.** Pobierz liste lokalizacji i wypisz ich laczna liczbe oraz
5 pierwszych nazw (`name`) po posortowaniu alfabetycznym.

**2.** Znajdz wszystkie regiony, ktorych `regionalDisplayName`
zawiera `(Europe)` i wypisz ich nazwy.

**3.** Sprawdz czy region `polandcentral` istnieje na liscie
dostepnych lokalizacji. Wypisz jego pelna nazwe (`displayName`)
lub komunikat `Nie znaleziono`. *(Trudniejsze)*

In [ ]:
# Cwiczenie 1 - liczba regionow i 5 pierwszych po sortowaniu
locations = az("account list-locations")
total = ...
sorted_names = ...

print(f"Laczna liczba regionow: {total}")
print("Pierwsze 5 (alfabetycznie):")
for name in sorted_names[:5]:
    print(f"  {name}")

In [ ]:
# Cwiczenie 2 - regiony europejskie po regionalDisplayName
locations = az("account list-locations")
# hint: sprawdz klucz 'regionalDisplayName' w kazdym elemencie
eu_regions = [
    loc["name"]
    for loc in locations
    if ...
]

print(f"Regiony europejskie ({len(eu_regions)}):")
for region in sorted(eu_regions):
    print(f"  {region}")

In [ ]:
# Cwiczenie 3 - sprawdzenie czy 'polandcentral' istnieje *(Trudniejsze)*
# hint: przeszukaj liste lokalizacji po kluczu 'name'
locations = az("account list-locations")
target = "polandcentral"

found = ...

if found:
    print(f"Znaleziono: {found['displayName']}")
else:
    print(f"Nie znaleziono regionu '{target}'")

---

## 5. 🔹 Tagowanie zasobow dla srodowisk testowych

Tagi (tags) to pary klucz-wartosc przypisywane do zasobow i
resource groups. Umozliwiaja organizacje, filtrowanie i
rozliczanie zasobow.

**Dlaczego QA powinna tagowac zasoby:**
- **filtrowanie** - `az resource list --tag env=test` pokazuje tylko
  zasoby testowe
- **koszty** - Cost Management pokazuje wydatki per projekt/srodowisko
- **czyszczenie** - skrypt usuwa wszystkie resource groups z
  tagiem `env=test` i `owner=qa-team` po zakonczeniu testow
- **identyfikacja** - kto i po co utworzyl zasob (audit)

**Konwencja tagowania dla QA:**

| Tag         | Przykladowe wartosci          | Opis                          |
|-------------|-------------------------------|-------------------------------|
| `env`       | `test`, `staging`, `prod`     | srodowisko                    |
| `owner`     | `qa-team`, `devops`           | odpowiedzialny zespol         |
| `project`   | `checkout-api`, `data-ingest` | projekt/aplikacja             |
| `module`    | `01`, `api-tests`             | modul lub typ testu           |
| `pr`        | `pr-123`, `main`              | numer pull requesta           |
| `ttl`       | `2026-04-10`                  | data wygasniecia (time-to-live)|

> 💡 Tagi `ttl` i `pr` sa szczegolnie uzyteczne w pipeline CI/CD -
> mozna napisac skrypt czyszczacy, ktory usuwa resource groups
> z `ttl` mniejszym niz dzisiejsza data lub `pr` zamknietym
> w repozytorium.

> ⚠️ Azure umozliwia max 50 tagow na zasob. Tagi nie sa
> dziedziczone - zasob w grupie nie dziedziczy tagow grupy.

In [ ]:
import datetime


def build_qa_tags(project, pr=None, module="01"):
    """Build standard QA tag set for Azure resources."""
    ttl = (datetime.date.today() + datetime.timedelta(days=1)).isoformat()
    tags = {
        "env": "test",
        "owner": "qa-team",
        "project": project,
        "module": module,
        "ttl": ttl,
    }
    if pr:
        tags["pr"] = pr
    return tags


tags = build_qa_tags("checkout-api", pr="pr-42")
print("Tagi dla nowego srodowiska testowego:")
for key, value in tags.items():
    print(f"  {key}: {value}")

In [ ]:
# Tworzenie resource group z tagami
# az group create przyjmuje --tags jako klucz=wartosc ...
def create_test_rg(name, location, tags):
    """Create resource group with QA tags."""
    tags_str = " ".join(f"{k}={v}" for k, v in tags.items())
    cmd = f"group create --name {name} --location {location} --tags {tags_str}"
    result = az(cmd)
    return result


tags = build_qa_tags("checkout-api", pr="pr-42")
rg = create_test_rg("rg-qa-checkout-pr42", "westeurope", tags)
print(f"Utworzono: {rg['name']}")
print(f"Tagi:      {rg['tags']}")

In [ ]:
# Listowanie resource groups z filtrem po tagu
# --query z filtrem JMESPath na polu tags
test_groups = az(
    "group list --query \"[?tags.env=='test']\""
)
print(f"Resource groups z tagiem env=test: {len(test_groups)}")
for rg in test_groups:
    print(f"  {rg['name']} | {rg['location']} | {rg['tags']}")

In [ ]:
# Czyszczenie po testach - usuwamy resource group po zakonczeniu
az("group delete --name rg-qa-checkout-pr42 --yes")
print("Srodowisko testowe usuniete.")

---

### 🐍 Cwiczenia - tagowanie zasobow

**1.** Uzupelnij funkcje `build_qa_tags(project, env, owner)`,
ktora zwraca slownik z tagami `env`, `owner`, `project` i `ttl`
(jutrzejsza data).

**2.** Utworz resource group `rg-qa-lab-<twoje-inicjaly>` z tagami
`env=test`, `owner=qa-team` i `project=azure-basics`.
Wypisz nazwe i tagi nowo powstalej grupy.

**3.** Pobierz liste resource groups i wypisz tylko te, ktore maja
tag `owner=qa-team`.

**4.** Napisz funkcje `cleanup_expired_rgs()`, ktora usuwa resource
groups, gdzie tag `ttl` jest wczesniejszy niz dzisiaj. *(Trudniejsze)*

In [ ]:
# Cwiczenie 1 - build_qa_tags z parametrami
import datetime


def build_qa_tags(project, env="test", owner="qa-team"):
    ttl = ...
    return {
        "env": ...,
        "owner": ...,
        "project": ...,
        "ttl": ...,
    }


tags = build_qa_tags("azure-basics")
print(tags)

In [ ]:
# Cwiczenie 2 - tworzenie resource group z tagami
# hint: zastap 'initials' swoimi inicjalami, np. 'kb'
rg_name = "rg-qa-lab-initials"
tags = build_qa_tags("azure-basics")
tags_str = " ".join(f"{k}={v}" for k, v in tags.items())

result = az(f"group create --name {rg_name} --location westeurope --tags {tags_str}")
print(f"Nazwa: {result['name']}")
print(f"Tagi:  {result['tags']}")

In [ ]:
# Cwiczenie 3 - resource groups z tagiem owner=qa-team
groups = az("group list")
qa_groups = [
    rg for rg in groups
    if ...
]

print(f"Resource groups z owner=qa-team: {len(qa_groups)}")
for rg in qa_groups:
    print(f"  {rg['name']}")

In [ ]:
# Cwiczenie 4 - cleanup_expired_rgs() *(Trudniejsze)*
# hint: porownaj tag 'ttl' (string YYYY-MM-DD) z datetime.date.today()
import datetime


def cleanup_expired_rgs():
    today = datetime.date.today()
    groups = az("group list")
    removed = []

    for rg in groups:
        ttl_str = rg.get("tags", {}).get("ttl")
        if ttl_str is None:
            continue
        ttl_date = ...
        if ...:
            print(f"Usuwam {rg['name']} (ttl: {ttl_str})")
            # az(f"group delete --name {rg['name']} --yes")
            removed.append(rg["name"])

    print(f"Usunieto {len(removed)} resource groups.")
    return removed


cleanup_expired_rgs()